# 06 — Cross-Dataset Evaluation and Model Comparison

Zero-shot Celeb-DF evaluation of each model from `04_model_baseline.ipynb` + `05_model_advanced.ipynb`, plus an ensemble, per-manipulation breakdown, ROC overlay, Grad-CAM visualization, and final leaderboard.

**Prerequisite:** Run this after `05_model_advanced.ipynb` has completed a full training pass on Colab Pro — it loads trained checkpoints from Drive. The 4 advanced models + the ResNet-18 baseline must already have rows in `experiments/results.csv`.

In [ ]:
import sys, torch, torch.nn as nn, pandas as pd, numpy as np
import torchvision.transforms as T
from datetime import datetime
from pathlib import Path
from torch.utils.data import DataLoader

from configs.paths import (REPO_ROOT, TRAIN_CSV, VAL_CSV, TEST_CSV,
                            FRAMES_ROOT, MTCNN_FRAMES_ROOT, RAFT_FRAMES_ROOT,
                            CELEBDF_FRAMES, CELEBDF_RAW_ROOT, MODEL_DIR,
                            RESULTS_CSV, RESULTS_JSON_DIR)
sys.path.insert(0, str(REPO_ROOT))

from src.datasets      import DeepfakeBinaryDataset
from src.models        import (ResNetBinaryVideoClassifier, EfficientNetDeepfakeDetector,
                                R3D18DeepfakeDetector, ViTDeepfakeDetector, R3D18RAFTDeepfakeDetector)
from src.training      import pick_device, load_checkpoint
from src.evaluation    import evaluate, per_manipulation_breakdown, cross_dataset_eval
from src.logging       import append_run_to_csv, write_run_json
from src.visualization import plot_roc_curves, plot_confusion_matrix, grad_cam_panel

device = pick_device()

# Pull the most recent run_id per model from experiments/results.csv
lb = pd.read_csv(RESULTS_CSV)
latest = lb.sort_values("timestamp").groupby("model").tail(1).set_index("model")
print(latest[["run_id","ffpp_test_acc","ffpp_test_f1","ffpp_test_auc"]])

IMAGENET_MEAN, IMAGENET_STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
KIN_MEAN, KIN_STD           = [0.43216,0.394666,0.37645], [0.22803,0.22145,0.216989]
eval_tfm_224 = T.Compose([T.ToPILImage(), T.Resize((224,224)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tfm_112 = T.Compose([T.ToPILImage(), T.Resize((112,112)), T.ToTensor(), T.Normalize(KIN_MEAN, KIN_STD)])

test_df = pd.read_csv(TEST_CSV)
FAKE_CLASSES = ["Deepfakes","Face2Face","FaceSwap","NeuralTextures","FaceShifter"]

## Celeb-DF v2 — zero-shot test set

Builds the Celeb-DF test DataFrame from the raw video folders. Extraction already happened in `03_preprocessing.ipynb`.

In [ ]:
celeb_rows = []
for label_name, target, folder in [("real",0,"Celeb-real"), ("real",0,"YouTube-real"), ("fake",1,"Celeb-synthesis")]:
    folder_path = CELEBDF_RAW_ROOT / folder
    if not folder_path.exists():
        print(f"warn: {folder_path} not found — skipping"); continue
    for vid in folder_path.rglob("*.mp4"):
        celeb_rows.append({"path": str(vid), "file": vid.name,
                           "binary_label": label_name, "binary_target": target,
                           "source_class": folder, "split": "test"})
celeb_df = pd.DataFrame(celeb_rows)
print("Celeb-DF test videos:", len(celeb_df))

## Zero-shot evaluation per model

Each model is loaded from its best checkpoint, then evaluated on both FF++ test (for a sanity check against the training-time numbers) and Celeb-DF test (zero-shot cross-dataset). Results are upserted into `experiments/results.csv` — `append_run_to_csv` has merge-semantics, so adding `celebdf_*` columns does not wipe the `ffpp_*` columns already written.

In [ ]:
MODELS = {
    # model_key: (Class, eval_transform, img_size, ffpp_frames_root, celebdf_frames_root)
    "resnet18":              (ResNetBinaryVideoClassifier,   eval_tfm_224, 224, FRAMES_ROOT,       CELEBDF_FRAMES),
    "efficientnet_b4":       (EfficientNetDeepfakeDetector,  eval_tfm_224, 224, MTCNN_FRAMES_ROOT, CELEBDF_FRAMES),
    "r3d18":                 (R3D18DeepfakeDetector,         eval_tfm_112, 112, MTCNN_FRAMES_ROOT, CELEBDF_FRAMES),
    "vit_base_patch16_224":  (ViTDeepfakeDetector,           eval_tfm_224, 224, MTCNN_FRAMES_ROOT, CELEBDF_FRAMES),
    "r3d18_raft":            (R3D18RAFTDeepfakeDetector,     eval_tfm_112, 112, RAFT_FRAMES_ROOT,  CELEBDF_FRAMES),
}

results = {}
for name, (Cls, tfm, _, ffpp_root, celeb_root) in MODELS.items():
    if name not in latest.index:
        print(f"skip {name}: no run in experiments/results.csv yet"); continue
    run_id = latest.loc[name, "run_id"]
    ckpt   = MODEL_DIR / "checkpoints" / run_id / f"{run_id}_best.pth"
    if not ckpt.exists():
        print(f"skip {name}: checkpoint not found at {ckpt}"); continue

    model = Cls()
    load_checkpoint(model, ckpt)
    model.to(device).eval()

    ffpp_ds   = DeepfakeBinaryDataset(test_df,  ffpp_root,  tfm, num_frames=16)
    celeb_ds  = DeepfakeBinaryDataset(celeb_df, celeb_root, tfm, num_frames=16)
    ffpp_loader  = DataLoader(ffpp_ds,  batch_size=8, num_workers=4, pin_memory=True)
    celeb_loader = DataLoader(celeb_ds, batch_size=8, num_workers=4, pin_memory=True)

    criterion = nn.CrossEntropyLoss()
    out = cross_dataset_eval(model, {"ffpp": ffpp_loader, "celebdf": celeb_loader}, criterion, device)
    gap = (out["ffpp"]["auc"] or 0) - (out["celebdf"]["auc"] or 0)
    results[name] = {"run_id": run_id, "ffpp": out["ffpp"], "celebdf": out["celebdf"], "gap": gap}

    # Upsert celebdf columns into the leaderboard (merge-semantics — preserves ffpp_* columns)
    append_run_to_csv(run_id, {"model": name}, {
        "celebdf_acc": out["celebdf"]["accuracy"],
        "celebdf_f1":  out["celebdf"]["f1"],
        "celebdf_auc": out["celebdf"]["auc"],
        "generalization_gap_auc": gap,
    }, RESULTS_CSV)
    print(f"{name:25s}  ffpp AUC={out['ffpp']['auc']:.4f}  celebdf AUC={out['celebdf']['auc']:.4f}  gap={gap:.4f}")

## Generalization-gap summary

The gap is FF++ test AUC minus Celeb-DF test AUC. A model with a small gap generalizes well across the two datasets; a large gap suggests the model is fitting FF++-specific artifacts.

In [ ]:
gap_rows = [{"model": n, "ffpp_auc": r["ffpp"]["auc"], "celebdf_auc": r["celebdf"]["auc"],
              "gap": r["gap"]} for n, r in results.items()]
gap_df = pd.DataFrame(gap_rows).sort_values("gap").reset_index(drop=True)
display(gap_df)

## Per-manipulation breakdown on FF++ test

Which fake types does each model handle best / worst? Breakdown by manipulation family (Deepfakes, Face2Face, FaceSwap, NeuralTextures, FaceShifter).

In [ ]:
breakdown_rows = []
for name, r in results.items():
    per_manip = per_manipulation_breakdown(test_df, r["ffpp"]["y_pred"], r["ffpp"]["y_probs"], FAKE_CLASSES)
    for manip, vals in per_manip.items():
        breakdown_rows.append({"model": name, "manip": manip,
                                "auc": vals["auc"], "f1": vals["f1"], "n_videos": vals["n_videos"]})
pd.DataFrame(breakdown_rows).pivot_table(index="manip", columns="model", values="auc")

## Ensemble (soft-vote across all 5 models)

Unweighted average of per-video fake-class probabilities. Simple but often beats any individual model on cross-dataset sets.

In [ ]:
# Only ensemble over models that have results
available = [n for n in MODELS if n in results]
ff_probs_stack    = np.mean([np.asarray(results[n]["ffpp"]["y_probs"])    for n in available], axis=0)
celeb_probs_stack = np.mean([np.asarray(results[n]["celebdf"]["y_probs"]) for n in available], axis=0)
ff_true    = np.asarray(results[available[0]]["ffpp"]["y_true"])
celeb_true = np.asarray(results[available[0]]["celebdf"]["y_true"])

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
ensemble_ffpp    = {"auc": roc_auc_score(ff_true, ff_probs_stack),
                    "f1":  f1_score(ff_true, (ff_probs_stack > 0.5).astype(int)),
                    "acc": accuracy_score(ff_true, (ff_probs_stack > 0.5).astype(int))}
ensemble_celebdf = {"auc": roc_auc_score(celeb_true, celeb_probs_stack),
                    "f1":  f1_score(celeb_true, (celeb_probs_stack > 0.5).astype(int)),
                    "acc": accuracy_score(celeb_true, (celeb_probs_stack > 0.5).astype(int))}
print("Ensemble FF++:    ", ensemble_ffpp)
print("Ensemble Celeb-DF:", ensemble_celebdf)

## ROC overlay — FF++ vs Celeb-DF

Models that retain their ROC shape across both datasets are the more generalization-friendly ones. Crossover points where one model's FF++ curve beats another's but loses on Celeb-DF are interesting to annotate.

In [ ]:
ffpp_plot  = {name: {"y_true": r["ffpp"]["y_true"],    "y_probs": r["ffpp"]["y_probs"]}    for name, r in results.items()}
celeb_plot = {name: {"y_true": r["celebdf"]["y_true"], "y_probs": r["celebdf"]["y_probs"]} for name, r in results.items()}
plot_roc_curves(ffpp_plot,  title="ROC — FF++ test").show()
plot_roc_curves(celeb_plot, title="ROC — Celeb-DF (zero-shot)").show()

## Grad-CAM — what does each model attend to?

Grad-CAM on EfficientNet-B4 (final conv stage) for 6 sample frames from the FF++ test set. Heatmap overlays indicate where the model looked to decide real vs. fake.

Skipped if EfficientNet-B4 checkpoint isn't available.

In [ ]:
if "efficientnet_b4" in results:
    from PIL import Image as _Image

    sample_rows = test_df.sample(6, random_state=42).reset_index(drop=True)
    sample_frames = []
    for _, row in sample_rows.iterrows():
        frame_dir = MTCNN_FRAMES_ROOT / row["binary_label"] / Path(row["file"]).stem
        jpgs = sorted(frame_dir.glob("*.jpg"))
        if not jpgs: continue
        first_frame = np.array(_Image.open(jpgs[0]))
        sample_frames.append(eval_tfm_224(first_frame))

    if sample_frames:
        sample_tensor = torch.stack(sample_frames).to(device)

        eff_run = latest.loc["efficientnet_b4", "run_id"]
        eff = EfficientNetDeepfakeDetector()
        load_checkpoint(eff, MODEL_DIR / "checkpoints" / eff_run / f"{eff_run}_best.pth")
        eff.to(device).eval()
        target_layer = eff.backbone.features[-1]   # last EfficientNet-B4 stage
        fig = grad_cam_panel(eff, sample_tensor, target_layer=target_layer,
                              denorm_mean=IMAGENET_MEAN, denorm_std=IMAGENET_STD)
        fig.show()
    else:
        print("No sample frames available for Grad-CAM — check MTCNN_FRAMES_ROOT is populated.")
else:
    print("EfficientNet-B4 checkpoint not in results — skipping Grad-CAM cell.")

## Final Leaderboard

Regenerated from `experiments/results.csv` — the single source of truth.

In [ ]:
lb_final = pd.read_csv(RESULTS_CSV).sort_values("timestamp").groupby("model").tail(1).reset_index(drop=True)
display(lb_final[["model","run_id","environment","ffpp_test_acc","ffpp_test_f1","ffpp_test_auc",
                   "celebdf_acc","celebdf_f1","celebdf_auc","generalization_gap_auc","train_time_minutes"]])

## Discussion — What Generalizes and Why

<!-- Abrar writes after Colab full runs:
- Which model shows the smallest FF++ → Celeb-DF AUC gap and why
- What the per-manipulation breakdown reveals about which fake types transfer
- What Grad-CAM suggests about spurious cues (background, skin tone) vs. artifact cues
- One or two paragraphs tying back to the literature (Rössler, Li, Wang) -->